# AUtoMLy: development evidence replay (October 18, 2026)

Recomputes every development number in Milestone 1 v5, Milestone 2 and Progress Log 2 from stored records. It makes no model calls, reads no final-test material and adds no human judgments. GPU steps (record generation, training, model answers) are not re-executed; their stored outputs are fingerprint-checked.

In [1]:
import collections, hashlib, importlib.util, json, runpy, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'CURRENT.md').exists():
    ROOT = ROOT.parent
B = ROOT / 'docs/engineering/document-builders'
sys.path.insert(0, str(B))
def mod(name, path):
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m); return m
v4 = mod('m1v4', B / 'build_milestone1_v4.py')
print('repo root:', ROOT)

repo root: <private source-of-record repository>


## 1. Recompute the derived development analysis and compare with the stored copy

In [2]:
OUT = ROOT / 'artifacts/milestone2_completion/class_package_20261018/development_analysis_20261018.json'
stored = json.loads(OUT.read_text())
runpy.run_path(str(B / 'derive_development_evidence_20261018.py'), run_name='__main__')
fresh = json.loads(OUT.read_text())
strip = lambda d: {k: v for k, v in d.items() if k not in ('generated_at', 'generated_at_central')}
assert strip(fresh) == strip(stored), 'recomputed analysis differs from stored copy'
print('derived analysis reproduced; sections:', [k for k in fresh if isinstance(fresh[k], (dict, list))])

{"written": "artifacts/milestone2_completion/class_package_20261018/development_analysis_20261018.json", "bundle_files": 6}
derived analysis reproduced; sections: ['input_sha256', 'probe', 'comparison', 'paired_base_vs_adapter', 'subgroup_doc_family', 'advisory_error_mix', 'human_review_nemotron', 'judge_reliability_check', 'latency_nemotron_batch_b', 'batch_a_dispositions']


## 2. Split and leakage checks (Milestone 1, Sections 1 and 6; Appendix B)

In [3]:
DATA = ROOT / 'artifacts/milestone1/2026-09-17/approved_candidates.jsonl'
CONTROL = ROOT / 'data/curated/student-authored-control-set-2026-09-22.jsonl'
rows, control = v4.load_rows(DATA), v4.load_rows(CONTROL)
print('dataset sha256', v4.sha(DATA)); print('controls sha256', v4.sha(CONTROL))
splits = collections.Counter(r['split'] for r in rows); print(len(rows), dict(splits))
docs = collections.defaultdict(set)
for r in rows: docs[(r['source_url'], r['source_sha256'])].add(r['split'])
print('documents per split', dict(collections.Counter(next(iter(s)) for s in docs.values())))
assert all(len(s) == 1 for s in docs.values()), 'a document appears in two splits'
ctrl = collections.Counter(tuple(sorted(docs[(c['source_url'], c['source_sha256'])])) for c in control)
print('control-question source splits', dict(ctrl)); assert ctrl == {('test',): 30}
groups = collections.defaultdict(list)
for r in rows: groups[v4.normal_answer(r['answer'])].append(r)
cross = [g for a, g in groups.items() if a and len({x['split'] for x in g}) > 1]
print('cross-split identical-answer groups', len(cross), 'records', sum(map(len, cross)))
assert (len(cross), sum(map(len, cross))) == (16, 57)

dataset sha256 f0d9324c68e7865f87cb1da3351d059b25ed08d2e08b9548073860ca7c9f702c
controls sha256 a1deb52304d7bce939593a228836debe2359488159e6e3b112e9d48ac7bea55c
1999 {'train': 1684, 'validation': 124, 'test': 191}
documents per split {'train': 113, 'validation': 10, 'test': 13}
control-question source splits {('test',): 30}
cross-split identical-answer groups 16 records 57


## 3. Training runs (Milestone 1, Section 2 and Appendix D)

In [4]:
COMP = ROOT / 'artifacts/model_comparison/2026-09-25'
gm, epoch_rows = v4.matched_run_summaries(v4.load_rows(COMP / 'results.jsonl'))
arch = v4.load(COMP / 'architecture-training-gates/clean-runs/architecture-clean-run-matrix.json')['runs']
runs = sorted({k for k in gm} | {(r['model_family'], r['train_size']) for r in arch})
n_epochs = len(epoch_rows) + sum(len(r['metrics_by_epoch']) for r in arch)
print('runs', len(runs), 'epoch records', n_epochs); assert (len(runs), n_epochs) == (16, 52)
for r in sorted(arch, key=lambda r: (r['model_family'], r['train_size'])):
    print(r['model_family'], r['train_size'], 'best', r['best_checkpoint'],
          'val loss by epoch', [round(e['eval_loss'], 4) for e in r['metrics_by_epoch']])

runs 16 epoch records 52
nemotron 200 best checkpoint-100 val loss by epoch [0.5859, 0.5825, 0.5862]
nemotron 509 best checkpoint-128 val loss by epoch [0.5614, 0.5637, 0.6118]
nemotron 750 best checkpoint-188 val loss by epoch [0.5574, 0.5593, 0.606]
nemotron 1600 best checkpoint-400 val loss by epoch [0.5527, 0.5549, 0.6001]
ornith 200 best checkpoint-50 val loss by epoch [0.6011, 0.6028, 0.6184]
ornith 509 best checkpoint-128 val loss by epoch [0.5822, 0.5829, 0.639]
ornith 750 best checkpoint-188 val loss by epoch [0.5809, 0.5834, 0.6355]
ornith 1600 best checkpoint-400 val loss by epoch [0.5626, 0.5797, 0.6254]


## 4. Development comparison, human review, judge check (M1 Tables 1–2; M2 Tables 1–4)

In [5]:
a = fresh
print(f"{'family':9} {'cond':8} {'fully supported':>16} {'95% CI':>12} {'unsupported':>12} {'abstained':>9} {'failed':>6}")
for r in a['comparison']:
    lo, hi = r['wilson95']
    print(f"{r['family']:9} {r['condition']:8} {r['fully_supported']:>13}/24 {lo:>6.0%}-{hi:<5.0%} "
          f"{r['unsupported_claims_present']:>9}/{r['judged']:<2} {r['abstained']:>9} {r['budget_failures'] + r['parser_failures']:>6}")
print('paired exact McNemar:', {f: v['exact_mcnemar_p'] for f, v in a['paired_base_vs_adapter'].items()})
h = a['human_review_nemotron']
print('Week 8 review strict:', h['base']['strict_correct_supported_no_unsupported'], '->',
      h['adapter']['strict_correct_supported_no_unsupported'], 'of 15; p =', h['paired']['exact_mcnemar_p'])
print('judge vs review agreement:', a['judge_reliability_check']['agreement'])
print('latency (s):', a['latency_nemotron_batch_b'])
for r in a['subgroup_doc_family']:
    if r['family'] == 'Nemotron': print('subgroup', r['condition'], r['doc_family'], r['fully_supported'], '/', r['slots'])
print('duplicate rulings:', a['batch_a_dispositions']['group_rulings'])
print('error mix, Nemotron:', a['advisory_error_mix']['Nemotron base'], a['advisory_error_mix']['Nemotron adapter'])
sys.path.insert(0, str(B)); import oct18_common; oct18_common.analysis(); print('report headline assertions pass')

family    cond      fully supported       95% CI  unsupported abstained failed
Nemotron  base                 4/24     7%-36%          17/24         3      0
Nemotron  adapter              4/24     7%-36%          19/24         0      0
Gemma     base                 0/24     0%-14%          10/24        14      0
Gemma     adapter              4/24     7%-36%          18/23         0      1
Ornith    base                 1/24     1%-20%           0/24        23      0
Ornith    adapter              4/24     7%-36%          18/23         0      1
Muse      base                 0/24     0%-14%           0/8          8     16
Muse      adapter              0/24     0%-14%           0/0          0     24
paired exact McNemar: {'Nemotron': 1.0, 'Gemma': 0.125, 'Ornith': 0.375, 'Muse': 1.0}
Week 8 review strict: 2 -> 4 of 15; p = 0.5
judge vs review agreement: {'factual_correctness': {'agree': 12, 'of': 12}, 'evidence_support': {'agree': 12, 'of': 12}, 'locator_validity': {'agree': 3, 'of':